# Fine-tuning Models on SageMaker

This notebook demonstrates how to fine-tune a pre-trained model on Amazon SageMaker to improve its performance on a specific task. Fine-tuning is a powerful technique that allows you to adapt a pre-trained model to your specific use case, often resulting in significant performance improvements.

## What is Fine-tuning?

Fine-tuning is the process of taking a model that has been pre-trained on a large dataset and then further training it on a smaller, task-specific dataset. This allows the model to adapt its learned features to the specific characteristics of your task.

Benefits of fine-tuning include:
- Improved accuracy on domain-specific tasks
- Faster training compared to training from scratch
- Better generalization with smaller datasets
- Adaptation to specific language patterns or terminology

## How Fine-tuning Complements Other Optimization Techniques

In previous notebooks, we explored various model optimization techniques:
- **Quantization**: Reducing the precision of model weights
- **Pruning**: Removing unnecessary connections in the model
- **Knowledge Distillation**: Creating smaller student models that learn from larger teacher models

Fine-tuning complements these techniques by:
1. Improving model quality before applying other optimizations
2. Recovering performance that might be lost during optimization
3. Adapting optimized models to specific domains

## What We'll Cover

In this notebook, we will:
1. Evaluate a pre-trained model on a sentiment analysis task
2. Prepare a dataset for fine-tuning
3. Configure and run a fine-tuning job on SageMaker
4. Deploy and evaluate the fine-tuned model
5. Compare performance before and after fine-tuning

Let's get started!

## Setup

First, let's install the specific packages needed for this fine-tuning notebook.

In [ ]:
# Install specific versions known to work together for fine-tuning
!pip install "torch==1.13.1" "transformers==4.26.0" "datasets==2.10.1" "accelerate==0.18.0" --quiet

In [ ]:
# Import common modules
from common_imports import *

# Import specific modules for this notebook


Now, let's import the necessary libraries and set up our SageMaker environment.

In [ ]:
import os
import json
import boto3
import sagemaker
import numpy as np
import pandas as pd
from sagemaker.huggingface import HuggingFace
from sagemaker.huggingface.model import HuggingFaceModel
import torch
import time
from datetime import datetime

# Import transformers components
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

In [ ]:
# Load workshop configuration
with open('workshop_config.json', 'r') as f:
    workshop_config = json.load(f)

# Set up SageMaker session
sagemaker_session = sagemaker.Session()
role = workshop_config['role']
region = workshop_config['region']
bucket = workshop_config['s3_bucket']
prefix = workshop_config['s3_prefix']

print(f"SageMaker session established in region: {region}")
print(f"Using S3 bucket: {bucket}")
print(f"Using S3 prefix: {prefix}")


## Model Selection

For this demonstration, we'll use DistilBERT, a smaller and faster version of BERT that maintains most of its performance. DistilBERT is a good choice for fine-tuning because:

1. It's smaller and faster than BERT, making it more suitable for production deployments
2. It's been pre-trained on a large corpus and can be fine-tuned effectively
3. It aligns with our workshop's focus on model optimization

We'll use a pre-trained DistilBERT model for sentiment analysis.

In [ ]:
# Define model ID
model_id = "distilbert-base-uncased"
task = "sentiment-analysis"

# Load pre-trained tokenizer and model for evaluation
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)

print(f"Model: {model_id}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters())}")

## Model Evaluation Before Fine-tuning

Before fine-tuning, let's evaluate how the pre-trained model performs on our target task: sentiment analysis for product reviews. We'll use a set of example reviews to demonstrate the model's performance.

Since the base DistilBERT model hasn't been specifically trained for sentiment analysis, we expect it to perform poorly on this task.

In [ ]:
# Create a sentiment analysis pipeline with the base model
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer)

# Example product reviews with expected sentiment
example_reviews = [
    {"text": "This product is amazing! I use it every day and it works perfectly.", "expected": "POSITIVE"},
    {"text": "Terrible quality, broke after just one week of use.", "expected": "NEGATIVE"},
    {"text": "The battery life is much shorter than advertised.", "expected": "NEGATIVE"},
    {"text": "Great value for the price, would definitely recommend.", "expected": "POSITIVE"},
    {"text": "The interface is confusing and not user-friendly.", "expected": "NEGATIVE"},
    {"text": "This smart device has transformed how I manage my home office setup.", "expected": "POSITIVE"},
    {"text": "Product arrived damaged and customer service was unhelpful.", "expected": "NEGATIVE"},
    {"text": "Average performance, nothing special but gets the job done.", "expected": "NEUTRAL"}
]

# Evaluate the model on these examples
print("\nPre-trained Model Predictions (Before Fine-tuning):")
print("-" * 60)
print(f"{'Review':<40} | {'Expected':<10} | {'Predicted':<10} | {'Match':<5}")
print("-" * 60)

correct = 0
for example in example_reviews:
    # Get prediction
    result = classifier(example["text"])[0]
    label = "POSITIVE" if result["label"] == "LABEL_1" else "NEGATIVE"
    
    # Check if prediction matches expected sentiment
    match = "✓" if label == example["expected"] else "✗"
    if match == "✓":
        correct += 1
    
    # Print result
    review_short = example["text"][:37] + "..." if len(example["text"]) > 40 else example["text"]
    print(f"{review_short:<40} | {example['expected']:<10} | {label:<10} | {match:<5}")

accuracy = correct / len(example_reviews) * 100
print("-" * 60)
print(f"Accuracy: {accuracy:.1f}%")

As we can see, the pre-trained model doesn't perform well on our sentiment analysis task. This is expected since the base DistilBERT model wasn't specifically trained for sentiment classification.

Next, we'll fine-tune this model on a sentiment analysis dataset to improve its performance.

## Dataset Preparation

For fine-tuning, we'll use the SST-2 (Stanford Sentiment Treebank) dataset, which is a popular benchmark for sentiment analysis. This dataset contains movie reviews labeled as positive or negative.

We'll load the dataset using the Hugging Face `datasets` library, prepare it for training, and upload it to S3 for SageMaker to access.

In [ ]:
# Load the SST-2 dataset
dataset = load_dataset("glue", "sst2")
print(f"Dataset loaded: {dataset}")

# Display dataset information
print(f"\nTraining examples: {len(dataset['train'])}")
print(f"Validation examples: {len(dataset['validation'])}")

# Display a few examples
print("\nSample examples:")
for i in range(3):
    example = dataset['train'][i]
    sentiment = "Positive" if example['label'] == 1 else "Negative"
    print(f"Example {i+1}: {example['sentence']} - {sentiment}")

Now, let's prepare the dataset for SageMaker training. We need to:
1. Convert the dataset to the format expected by the training script
2. Split it into training and validation sets
3. Upload it to S3

In [ ]:
# Create directories for the dataset
os.makedirs("data", exist_ok=True)

# Prepare the training data
train_data = []
for example in dataset["train"]:
    train_data.append({
        "text": example["sentence"],
        "label": example["label"]
    })

# Prepare the validation data
validation_data = []
for example in dataset["validation"]:
    validation_data.append({
        "text": example["sentence"],
        "label": example["label"]
    })

# Save the data to disk
with open("data/train.json", "w") as f:
    for item in train_data:
        f.write(json.dumps(item) + "\n")

with open("data/validation.json", "w") as f:
    for item in validation_data:
        f.write(json.dumps(item) + "\n")

print(f"Saved {len(train_data)} training examples to data/train.json")
print(f"Saved {len(validation_data)} validation examples to data/validation.json")

In [ ]:
# Upload the dataset to S3
train_s3_uri = sagemaker_session.upload_data(
    path="data/train.json",
    bucket=default_bucket,
    key_prefix=f"{prefix}/data/train"
)

validation_s3_uri = sagemaker_session.upload_data(
    path="data/validation.json",
    bucket=default_bucket,
    key_prefix=f"{prefix}/data/validation"
)

print(f"Training data uploaded to: {train_s3_uri}")
print(f"Validation data uploaded to: {validation_s3_uri}")

## Fine-tuning on SageMaker

Now that we have prepared our dataset, let's configure and run a fine-tuning job on SageMaker. We'll use the Hugging Face Estimator provided by the SageMaker Python SDK, which makes it easy to train Hugging Face models on SageMaker.

In [ ]:
# Define the hyperparameters for fine-tuning
hyperparameters = {
    'model_name_or_path': model_id,
    'output_dir': '/opt/ml/model',
    'overwrite_output_dir': True,
    'per_device_train_batch_size': 16,
    'per_device_eval_batch_size': 16,
    'num_train_epochs': 3,
    'learning_rate': 5e-5,
    'warmup_steps': 500,
    'weight_decay': 0.01,
    'logging_dir': '/opt/ml/output/tensorboard',
    'logging_steps': 100,
    'evaluation_strategy': 'epoch',
    'save_strategy': 'epoch',
    'load_best_model_at_end': True,
    'metric_for_best_model': 'eval_loss',
    'greater_is_better': False,
    'seed': 42
}

In [ ]:
# Configure the SageMaker HuggingFace estimator
huggingface_estimator = HuggingFace(
    entry_point='train_sentiment.py',
    source_dir='scripts',
    instance_type='ml.g4dn.xlarge',  # Use a GPU instance for faster training
    instance_count=1,
    role=role,
    transformers_version='4.26.0',
    pytorch_version='1.13.1',
    py_version='py39',
    hyperparameters=hyperparameters,
    disable_profiler=True,
    debugger_hook_config=False
)

In [ ]:
# Define the data channels
data_channels = {
    'train': train_s3_uri,
    'validation': validation_s3_uri
}

# Start the training job
print("Starting fine-tuning job...")
huggingface_estimator.fit(data_channels, wait=False)
print(f"Training job name: {huggingface_estimator.latest_training_job.job_name}")

In [ ]:
# Wait for the training job to complete
print("Training in progress. This may take 15-30 minutes...")
huggingface_estimator.latest_training_job.wait(logs="None")
print("Training job completed!")

# Get the model artifacts
model_artifacts = huggingface_estimator.model_data
print(f"Model artifacts stored at: {model_artifacts}")

## Deploying the Fine-tuned Model

After fine-tuning, we'll deploy the model to a SageMaker endpoint for inference.

In [ ]:
# Configure the model
huggingface_model = HuggingFaceModel(
    model_data=model_artifacts,
    role=role,
    transformers_version='4.26.0',
    pytorch_version='1.13.1',
    py_version='py39',
)

In [ ]:
# Deploy the model to an endpoint
print("Deploying model to endpoint...")
predictor = huggingface_model.deploy(
    initial_instance_count=1,
    instance_type='ml.g4dn.xlarge',
)
print(f"Model deployed to endpoint: {predictor.endpoint_name}")

## Evaluating the Fine-tuned Model

Now let's evaluate the fine-tuned model on our test examples to see how much the performance has improved.

In [ ]:
# Save the pre-trained model accuracy for comparison
pre_trained_accuracy = accuracy

# Prepare test examples for inference
test_examples = [example["text"] for example in example_reviews]

# Get predictions from the deployed model
print("Getting predictions from fine-tuned model...")
predictions = predictor.predict({
    "inputs": test_examples
})

In [ ]:
# Process and display the results
print("\nFine-tuned Model Predictions (After Fine-tuning):")
print("-" * 60)
print(f"{'Review':<40} | {'Expected':<10} | {'Predicted':<10} | {'Match':<5}")
print("-" * 60)

correct = 0
for i, example in enumerate(example_reviews):
    # Get prediction
    prediction = predictions[i]
    label = "POSITIVE" if prediction["label"] == "LABEL_1" else "NEGATIVE"
    
    # Check if prediction matches expected sentiment
    match = "✓" if label == example["expected"] else "✗"
    if match == "✓":
        correct += 1
    
    # Print result
    review_short = example["text"][:37] + "..." if len(example["text"]) > 40 else example["text"]
    print(f"{review_short:<40} | {example['expected']:<10} | {label:<10} | {match:<5}")

accuracy = correct / len(example_reviews) * 100
print("-" * 60)
print(f"Accuracy: {accuracy:.1f}%")

## Comparing Performance

Let's compare the performance of the pre-trained model and the fine-tuned model.

In [ ]:
# Print comparison
print("\nPerformance Comparison:")
print("-" * 60)
print(f"Pre-trained model accuracy: {pre_trained_accuracy:.1f}%")
print(f"Fine-tuned model accuracy: {accuracy:.1f}%")
print(f"Improvement: {accuracy - pre_trained_accuracy:.1f}%")

## Cleanup

Finally, let's clean up the resources we created.

In [ ]:
# Delete the endpoint
print("Cleaning up resources...")
predictor.delete_endpoint()
print("Endpoint deleted.")

## Conclusion

In this notebook, we demonstrated how to fine-tune a pre-trained model on SageMaker to improve its performance on a specific task. We saw that fine-tuning significantly improved the model's accuracy on our sentiment analysis task.

Fine-tuning is a powerful technique that complements other optimization methods like quantization, pruning, and knowledge distillation. By combining these techniques, you can create models that are both accurate and efficient.

Key takeaways:
1. Fine-tuning adapts pre-trained models to specific tasks
2. SageMaker makes it easy to run fine-tuning jobs at scale
3. Fine-tuned models can achieve significantly better performance than pre-trained models
4. Fine-tuning can be combined with other optimization techniques for even better results